# PanNuke × Classpose — Data Exploration

Validates the converted `.npy` files before training.

**What we check:**
1. Sample image + overlaid instance/class maps
2. Class distribution across all images
3. Nucleus-free patches (all-background labels)
4. Fold boundaries


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

DATA_TRAIN = "data/train"
DATA_TEST  = "data/test"

CLASS_NAMES = {
    0: "background",
    1: "Neoplastic",
    2: "Inflammatory",
    3: "Connective",
    4: "Dead",
    5: "Epithelial",
}
# Colorblind-friendly palette (skip index 0 = background = black)
CLASS_COLORS = ["black", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#CC79A7"]
CMAP = ListedColormap(CLASS_COLORS)

images = np.load(f"{DATA_TRAIN}/images.npy", mmap_mode="r")
labels = np.load(f"{DATA_TRAIN}/labels.npy", mmap_mode="r")

print(f"images: {images.shape}  dtype={images.dtype}")
print(f"labels: {labels.shape}  dtype={labels.dtype}")

with open(f"{DATA_TRAIN}/fold_boundaries.json") as f:
    boundaries = json.load(f)
print("\nFold boundaries:", boundaries)


## Sample image + label overlay

In [ ]:
def show_sample(idx):
    img  = images[idx]                   # (256, 256, 3)
    inst = labels[idx, :, :, 0]          # instance map
    cls  = labels[idx, :, :, 1]          # class map (1-indexed)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(f"Index {idx}  |  {inst.max()} nuclei", fontsize=13)

    axes[0].imshow(img)
    axes[0].set_title("H&E image")
    axes[0].axis("off")

    axes[1].imshow(inst, cmap="nipy_spectral", interpolation="nearest")
    axes[1].set_title("Instance map")
    axes[1].axis("off")

    cls_plot = axes[2].imshow(cls, cmap=CMAP, vmin=0, vmax=5, interpolation="nearest")
    axes[2].set_title("Class map")
    axes[2].axis("off")
    patches = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_NAMES[c]) for c in range(6)]
    axes[2].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)

    plt.tight_layout()
    plt.show()

# Show a few examples
for idx in [0, 100, 500]:
    show_sample(idx)


## Class distribution

In [ ]:
cls_map = labels[:, :, :, 1]   # (N, 256, 256)

print("Pixel counts per class:")
pixel_counts = {}
for c in range(6):
    n = int((cls_map == c).sum())
    pixel_counts[c] = n
    print(f"  {c} {CLASS_NAMES[c]:<18} {n:>12,}")

# Instance counts (faster than pixel-level: count unique IDs in inst map)
inst_map = labels[:, :, :, 0]
print("\nNuclei (instance) counts per class:")
for c in range(1, 6):
    # For each image, get unique inst IDs where class == c
    total = 0
    for i in range(len(labels)):
        ids = np.unique(inst_map[i][cls_map[i] == c])
        total += int((ids > 0).sum())
    print(f"  {c} {CLASS_NAMES[c]:<18} {total:>8,}")

# Bar chart of nucleus pixel counts (excluding background)
fg_counts = [pixel_counts[c] for c in range(1, 6)]
fg_names  = [CLASS_NAMES[c] for c in range(1, 6)]
colors    = [CLASS_COLORS[c] for c in range(1, 6)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(fg_names, fg_counts, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Pixel count")
ax.set_title("Class pixel distribution (foreground only)")
for bar, n in zip(bars, fg_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f"{n/1e6:.1f}M", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## Nucleus-free patches & label sanity checks

In [ ]:
inst_map = labels[:, :, :, 0]

# Images with no nuclei at all
n_bg_only = int((inst_map.max(axis=(1, 2)) == 0).sum())
print(f"Nucleus-free patches: {n_bg_only} / {len(images)}  ({100*n_bg_only/len(images):.1f}%)")

# Class map == 0 everywhere (should match bg-only count)
cls_map_all_bg = int((cls_map.max(axis=(1, 2)) == 0).sum())
print(f"Class-map all-background: {cls_map_all_bg} / {len(images)}")

# Confirm no pixel where inst>0 but cls==0 (would indicate a conversion bug)
mismatch = int(((inst_map > 0) & (cls_map == 0)).sum())
print(f"Pixels with instance>0 but class==0 (should be 0): {mismatch}")

# Max instances per image
max_per_img = inst_map.max(axis=(1, 2))
print(f"\nNuclei per image — min: {max_per_img.min()}  median: {int(np.median(max_per_img))}  max: {max_per_img.max()}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(max_per_img, bins=50, color="#56B4E9", edgecolor="black", linewidth=0.4)
ax.set_xlabel("Nuclei per image")
ax.set_ylabel("Count")
ax.set_title("Distribution of nucleus count per image")
plt.tight_layout()
plt.show()
